Notebook to compute flux bias of HCDs depending on cosmology, cddf and redshift

In [1]:
from lace.cosmo import cosmology
import numpy as np
import h5py
from model import compute_w1_w2
from pyigm.fN.fnmodel import FNModel

-----------------------------------------------------------
-----------------------------------------------------------
 Install pymc if you want it
-----------------------------------------------------------


First we need to set the cosmology

In [2]:
# ASTRID cosmology, for example
hubble, z = 0.6774, 2.5
ombh2, omegam = 0.0486*(hubble**2), 0.3089
omch2 = omegam*(hubble**2)-ombh2
ns, As = 0.9667, 2.142e-9
Fhcd = 0.9779  # Mean HCD flux, value taken from ASTRID simulation

cosmo = cosmology.Cosmology()
cosmo.CAMBparams.set_cosmology(H0=100*hubble, ombh2=ombh2, omch2=omch2)
cosmo.CAMBparams.InitPower.set_params(As=As, ns=ns);

Setting large-scale bias on HCDs and flux bias on lya absorption

In [3]:
bias_hcd, bias_lya = 2, -0.18

Setting NHi range to be evaluated and calling pyigm to obtain f(Nhi, X) (This model extrapolates points from Prochaska14)

In [5]:
NHirange = np.linspace(17.22, 22.3) 
fN_default = FNModel.default_model(cosmo=cosmo)
log_fNHiX = fN_default.evaluate(NHI=NHirange, z=z, cosmo=cosmo)
fNHiX = 10**log_fNHiX

Using P14 spline values to generate a default model
Loading: /data/desi/scratch/lfloresl/hcd_env/pyigm/pyigm/data/fN/fN_spline_z24.fits.gz


/data/desi/scratch/lfloresl/hcd_env/pyigm/pyigm/fN/fnmodel.py:66: UserWarning: Be careful here.  This fN model was derived with a specific cosmology.
  warnings.warn("Be careful here.  This fN model was derived with a specific cosmology.")
        Use numpy.iterable() instead. [pyigm.fN.fnmodel]
        Use numpy.iterable() instead. [pyigm.fN.fnmodel]
        Use numpy.iterable() instead. [pyigm.fN.fnmodel]


Computing w1 and w2

In [13]:
wave_log = np.linspace(np.log10(2800), np.log10(6800), 2**12) # Angstrom
wave = 10**wave_log
k_w1w2, w1, w2 = compute_w1_w2(wave, NHirange, fNHiX, z, hubble, omegam, resample=True)

Computing HCD flux bias as:

In [14]:
bias_hcd_flux = -2*w2[1]/Fhcd  # Warning, w2 depends a lot on the cosmology
print(bias_hcd_flux)

-0.041222852672404296
